# MYELIN-SR Phase 2: Temporal Training

Trains MYELIN-Flow and TemporalFusion on the REDS dataset using the frozen Phase 1 spatial brain.

In [ ]:
# SETUP AND IMPORTS
import os
import time
import math
import random
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

# Ensure reproducibility
torch.manual_seed(42)
random.seed(42)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_count = torch.cuda.device_count()
print(f"GPUs detected:   {gpu_count}")


## Core Architecture (From Phase 1)

In [ ]:
"""
FPSANConv2d â€” The Core Building Block of MYELIN-SR

Extends Krishna's FPSANLinear (core_dump/FP-SAN/src/fpsan_proofsheet.py) to 2D convolutions.

Key Mechanics:
  1. TERNARY FORWARD PASS: Weights are FP32 during training but quantized to {-1, 0, +1}
     inside the forward pass via round(w/scale).clamp(-1,1)*scale. Gradients flow through
     the full-precision master weights (Straight-Through Estimator).

  2. ASTROCYTIC STIFFNESS: Per-weight consolidation buffer that tracks activity.
     Heavily-used weights become "stiff" (protected from future updates), solving
     catastrophic forgetting. This is the FP-SAN equivalent of EWC but simpler.

  3. SLEEP CONSOLIDATION: Converts accumulated daily_activity into permanent stiffness,
     then resets activity. Hardware equivalent of biological memory consolidation.

  4. DYNAMIC GROWTH: When average stiffness exceeds a threshold (default 0.85),
     the layer spawns a GrowthModule â€” a fresh parallel learner that handles patterns
     the frozen parent cannot. Growth is recursive: a child can grow its own child.
     This means learning NEVER stops, regardless of how consolidated the network is.

  5. MULTIPLICATION-FREE INFERENCE: At export time, weights are quantized to int8 {-1,0,1}.
     C++/HLSL inference uses only add/subtract branching â€” zero FP multiplications.

Novel contribution (not in literature):
  - Recursive GrowthModule trees in ternary quantized networks
  - Stiffness-gated neuron spawning as a continual learning mechanism
  - DFA updates that respect stiffness gradients

References:
  - FPSANLinear: core_dump/FP-SAN/src/fpsan_proofsheet.py L105-L148
  - Export pipeline: core_dump/FP-SAN/src/export_weights.py
  - C++ inference: core_dump/FP-SAN/src/fpsan_infer.cpp L62-L96
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional


class FPSANConv2d(nn.Module):
    """
    Ternary-Quantized 2D Convolution with Astrocytic Stiffness and Dynamic Growth.

    Drop-in replacement for nn.Conv2d. Trains in FP32, forward-passes in ternary.
    Supports depthwise separable mode via groups parameter.

    NEW: When stiffness saturates (avg > growth_threshold), spawns a GrowthModule
    â€” a fresh parallel ternary learner that adds to this layer's output.
    Growth is recursive: children can grow their own children when they too saturate.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        groups: int = 1,
        bias: bool = True,
        min_plasticity: float = 0.15,
        stiffness_cap: float = 6.0,
        growth_threshold: float = 0.85,   # stiffness fraction to trigger growth
        growth_channels: int = 0,          # 0 = auto (25% of out_channels, min 4)
        max_growth_depth: int = 3,         # prevent infinite recursion
        _depth: int = 0,                   # internal: current recursion depth
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size if isinstance(kernel_size, tuple) else (kernel_size, kernel_size)
        self.stride = stride
        self.padding = padding
        self.groups = groups
        self.min_plasticity = min_plasticity
        self.stiffness_cap = stiffness_cap
        self.growth_threshold = growth_threshold
        self._growth_channels = growth_channels
        self.max_growth_depth = max_growth_depth
        self._depth = _depth

        # FP32 master weights (training state)
        self.weight = nn.Parameter(
            torch.empty(out_channels, in_channels // groups, *self.kernel_size)
        )
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_channels))
        else:
            self.register_parameter("bias", None)

        # Astrocytic Stiffness: per-weight consolidation buffer
        self.register_buffer(
            "stiffness", torch.ones_like(self.weight.data)
        )
        # Daily activity tracker (accumulated between sleep phases)
        self.register_buffer(
            "daily_activity", torch.zeros_like(self.weight.data)
        )

        # Input trace for forward-mode learning (DFA)
        self.trace_input: Optional[torch.Tensor] = None

        # Growth child â€” spawned lazily when stiffness saturates
        # Registered as a submodule so it appears in state_dict and parameters()
        self.growth_child: Optional['FPSANConv2d'] = None

        # Gate scale for growth child output â€” starts at 0, learned up
        # This ensures the child starts with ZERO influence and grows gradually
        self.register_buffer("growth_gate", torch.zeros(1))

        # Initialize weights (Kaiming uniform, as standard for conv layers)
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in = in_channels // groups * self.kernel_size[0] * self.kernel_size[1]
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)

    @property
    def _auto_growth_channels(self) -> int:
        """Auto-compute growth channels: 25% of out_channels, minimum 4."""
        if self._growth_channels > 0:
            return self._growth_channels
        return max(4, self.out_channels // 4)

    def _quantize_ternary(self, weight: torch.Tensor) -> torch.Tensor:
        """
        Ternary quantization: round to {-1, 0, +1} scaled by mean absolute value.

        This is Krishna's original technique from FPSANLinear:
            scale = weight.abs().mean().clamp_min(1e-5)
            w_quant = round(weight / scale).clamp(-1, 1) * scale

        During training, gradients flow through the FP32 master weights via
        the Straight-Through Estimator (STE) â€” the quantization is treated
        as identity in the backward pass.
        """
        scale = weight.detach().abs().mean().clamp_min(1e-5)
        # Forward: quantized. Backward: straight-through to FP32 weights.
        w_scaled = weight / scale
        w_ternary = (torch.round(w_scaled).clamp(-1, 1) - w_scaled).detach() + w_scaled
        return w_ternary * scale

    def get_avg_stiffness_fraction(self) -> float:
        """Return normalized average stiffness in [0, 1]."""
        return (self.stiffness / self.stiffness_cap).mean().item()

    def is_saturated(self) -> bool:
        """True if this layer's neurons are mostly frozen."""
        return self.get_avg_stiffness_fraction() >= self.growth_threshold

    def spawn_growth_child(self) -> bool:
        """
        Spawn a fresh parallel ternary learner if stiffness is saturated and
        max depth not reached.

        The child has the same in/out channels and kernel size, but:
          - Starts with near-zero output influence (growth_gate=0)
          - Has its own fresh stiffness (all ones initially, low accumulated)
          - Can itself spawn a grandchild when it saturates

        Returns True if a child was spawned, False if already exists or max depth.
        """
        if self.growth_child is not None:
            # Try growing the existing child recursively
            return self.growth_child.spawn_growth_child()

        if self._depth >= self.max_growth_depth:
            return False

        n = self._auto_growth_channels

        # Child handles the residual in the same feature space
        # We use a 1x1 bottleneck first to reduce params, then match out_channels
        self.growth_child = FPSANConv2d(
            in_channels=self.in_channels,
            out_channels=self.out_channels,
            kernel_size=self.kernel_size[0],
            stride=self.stride,
            padding=self.padding,
            groups=self.groups,
            bias=True,
            min_plasticity=self.min_plasticity,
            stiffness_cap=self.stiffness_cap,
            growth_threshold=self.growth_threshold,
            growth_channels=max(4, n // 2),    # children grow smaller
            max_growth_depth=self.max_growth_depth,
            _depth=self._depth + 1,
        )

        # Initialize child weights small: start nearly silent
        with torch.no_grad():
            nn.init.kaiming_uniform_(self.growth_child.weight, a=math.sqrt(5))
            self.growth_child.weight.data.mul_(0.01)  # 100x smaller initial magnitude
            if self.growth_child.bias is not None:
                self.growth_child.bias.data.zero_()

        # Slowly open gate: will be learned upward via gradient
        self.growth_gate.fill_(0.0)

        return True

    def check_and_grow(self) -> bool:
        """
        Called periodically (e.g., during sleep consolidation).
        Spawns a growth child if stiffness is saturated and none exists yet.
        If child exists, checks if IT needs to grow recursively.
        """
        if self.is_saturated():
            spawned = self.spawn_growth_child()
            return spawned
        return False

    def get_growth_depth(self) -> int:
        """Return how many growth children have been spawned (depth of tree)."""
        if self.growth_child is None:
            return 0
        return 1 + self.growth_child.get_growth_depth()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Store input trace for potential DFA update
        if self.training:
            self.trace_input = x.detach()

        # Quantize weights to ternary in forward pass
        w_q = self._quantize_ternary(self.weight)

        out = F.conv2d(
            x, w_q, self.bias,
            stride=self.stride,
            padding=self.padding,
            groups=self.groups,
        )

        # Add growth child output if spawned â€” gated to start near-zero
        if self.growth_child is not None:
            child_out = self.growth_child(x)
            # growth_gate is learned: starts at 0, increases as child proves useful
            gate = torch.sigmoid(self.growth_gate)
            out = out + gate * child_out

        return out

    @torch.no_grad()
    def apply_forward_update(
        self,
        local_error: torch.Tensor,
        lr: float,
        sample_gate: Optional[torch.Tensor] = None,
    ) -> None:
        """
        Direct Feedback Alignment (DFA) update â€” backprop-free.

        Uses the stored input trace and a local error signal to update weights
        without computing a full backward pass through the network.

        For convolutional layers, this computes the correlation between input
        patches and error signals, gated by astrocytic stiffness.

        References: fpsan_proofsheet.py L124-L142
        """
        if self.trace_input is None:
            return

        # Compute weight gradient via correlation of input and error
        # For conv layers, we use F.conv2d in "backward" configuration
        batch_size = local_error.size(0)

        # Unfold input to get patches matching the kernel
        # This is equivalent to the gradient computation for conv2d
        x = self.trace_input
        if sample_gate is not None:
            gate = sample_gate.view(batch_size, 1, 1, 1)
            local_error = local_error * gate
            normalizer = gate.sum().clamp_min(1.0)
        else:
            normalizer = float(batch_size)

        # Compute raw weight update (gradient approximation)
        # Using the efficient method: grad_w = conv2d(input^T, error^T)
        raw_weight_grad = torch.zeros_like(self.weight)
        for b in range(batch_size):
            # Per-sample gradient accumulation (memory-efficient)
            x_b = x[b:b+1]
            e_b = local_error[b:b+1]
            # Standard convolution gradient formula
            for g in range(self.groups):
                c_in = self.in_channels // self.groups
                c_out = self.out_channels // self.groups
                x_g = x_b[:, g*c_in:(g+1)*c_in]
                e_g = e_b[:, g*c_out:(g+1)*c_out]
                for co in range(c_out):
                    for ci in range(c_in):
                        raw_weight_grad[g*c_out+co, ci] += F.conv2d(
                            x_g[:, ci:ci+1],
                            e_g[:, co:co+1].flip(-1, -2),
                            padding=self.kernel_size[0] - 1
                        )[0, 0, :self.kernel_size[0], :self.kernel_size[1]]

        raw_weight_grad /= normalizer

        # Apply Astrocytic Stiffness gating
        normalized_stiffness = self.stiffness / self.stiffness.max().clamp_min(1.0)
        plasticity_mask = (1.0 - normalized_stiffness).clamp(self.min_plasticity, 1.0)

        # Update FP32 master weights (not the quantized ones)
        self.weight.data -= lr * (raw_weight_grad * plasticity_mask / self.stiffness.clamp_min(1.0))

        if self.bias is not None:
            raw_bias_grad = local_error.mean(dim=(0, 2, 3))
            self.bias.data -= lr * raw_bias_grad

        # Track activity for sleep consolidation
        self.daily_activity += raw_weight_grad.abs()
        self.trace_input = None

        # Propagate DFA signal to growth child with attenuated error
        if self.growth_child is not None:
            self.growth_child.apply_forward_update(
                local_error * 0.5,   # attenuate â€” child learns residual
                lr,
                sample_gate,
            )

    @torch.no_grad()
    def consolidate(self, rate: float = 0.02) -> None:
        """
        Sleep Consolidation: Convert daily activity into permanent stiffness.

        High-activity weights become stiffer (harder to modify), protecting
        learned features from catastrophic forgetting during new task learning.

        After consolidation, check if growth should be triggered.

        Reference: fpsan_proofsheet.py L144-L148
        """
        if rate <= 0:
            return
        self.stiffness.add_(self.daily_activity * rate).clamp_(max=self.stiffness_cap)
        self.daily_activity.zero_()

        # Cascade consolidation to growth child
        if self.growth_child is not None:
            self.growth_child.consolidate(rate)

    @torch.no_grad()
    def get_ternary_weights(self) -> torch.Tensor:
        """Export pure ternary {-1, 0, +1} int8 weights for C++/HLSL inference."""
        scale = self.weight.abs().mean().clamp_min(1e-5)
        return torch.round(self.weight / scale).clamp(-1, 1).to(torch.int8)

    @torch.no_grad()
    def get_weight_scale(self) -> float:
        """Get the per-layer scale factor for ternary reconstruction."""
        return self.weight.abs().mean().clamp_min(1e-5).item()

    def get_sparsity(self) -> float:
        """Return the fraction of zero (inactive) ternary weights."""
        w_ternary = self.get_ternary_weights()
        return (w_ternary == 0).float().mean().item()

    def extra_repr(self) -> str:
        s = (
            f"{self.in_channels}, {self.out_channels}, "
            f"kernel_size={self.kernel_size}, stride={self.stride}, "
            f"padding={self.padding}, groups={self.groups}"
        )
        sparsity = self.get_sparsity() * 100
        depth = self.get_growth_depth()
        saturated = "SAT" if self.is_saturated() else "OK"
        return s + f", sparsity={sparsity:.1f}%, stiffness={saturated}, growth_depth={depth}"


class FP16Conv2d(nn.Module):
    """
    Standard FP16 convolution for the color-precision-critical reconstruction head.

    Used only for the PixelShuffle reconstruction layer (Decision 3: Hybrid Precision).
    Everything else uses FPSANConv2d (ternary).
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        bias: bool = True,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size,
            stride=stride, padding=padding, bias=bias,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Use FP16 for precision-critical color reconstruction
        if x.is_cuda and x.dtype == torch.float32:
            with torch.amp.autocast("cuda"):
                return self.conv(x)
        return self.conv(x)


class NetworkGrowthManager:
    """
    Monitors all FPSANConv2d layers in a model and triggers dynamic growth
    when stiffness saturates. Provides reporting on the growth state.

    Usage:
        manager = NetworkGrowthManager(model, growth_threshold=0.85)
        # During training, call periodically:
        manager.step(epoch)  # checks saturation, spawns children
        manager.report()     # print growth status
    """

    def __init__(
        self,
        model: nn.Module,
        growth_threshold: float = 0.85,
        check_interval: int = 10,   # check every N epochs
        verbose: bool = True,
    ):
        self.model = model
        self.growth_threshold = growth_threshold
        self.check_interval = check_interval
        self.verbose = verbose
        self._total_spawned = 0

    def _get_fpsan_layers(self):
        """Yield (name, module) for all FPSANConv2d in model (not children of children)."""
        for name, module in self.model.named_modules():
            if isinstance(module, FPSANConv2d) and module._depth == 0:
                yield name, module

    def step(self, epoch: int) -> int:
        """
        Check all base FPSANConv2d layers for saturation and spawn growth children.
        Returns number of new children spawned this step.
        """
        if epoch % self.check_interval != 0:
            return 0

        spawned = 0
        for name, layer in self._get_fpsan_layers():
            if layer.check_and_grow():
                spawned += 1
                self._total_spawned += 1
                depth = layer.get_growth_depth()
                if self.verbose:
                    stiff_pct = layer.get_avg_stiffness_fraction() * 100
                    print(f"  [GROWTH] {name}: stiffness={stiff_pct:.1f}% "
                          f"-> spawned child (tree depth={depth})")

        return spawned

    def report(self) -> dict:
        """Print and return growth status for all base layers."""
        stats = {
            "total_spawned": self._total_spawned,
            "layers": {}
        }
        for name, layer in self._get_fpsan_layers():
            stiff = layer.get_avg_stiffness_fraction()
            depth = layer.get_growth_depth()
            stats["layers"][name] = {
                "stiffness_pct": stiff * 100,
                "saturated": layer.is_saturated(),
                "growth_depth": depth,
            }

        if self.verbose:
            saturated = sum(1 for v in stats["layers"].values() if v["saturated"])
            grown = sum(1 for v in stats["layers"].values() if v["growth_depth"] > 0)
            total = len(stats["layers"])
            print(f"\n[GrowthManager] {saturated}/{total} layers saturated, "
                  f"{grown}/{total} have grown, {self._total_spawned} total spawns")
        return stats


In [ ]:
"""
MYELIN-SRNet â€” The Super Resolution Architecture

Spike-Routed Mixture of Experts with Combinatorial Texture Encoding,
Hybrid Precision (ternary backbone + FP16 reconstruction), and
Astrocytic Stiffness for continual learning.

Architecture Overview:
  Input (LR) â†’ Shallow Feature Extraction (ternary)
    â†’ Texture State Encoder (8 neurons, 255 combo states)
    â†’ Spiking Router â†’ Expert Selection
    â†’ N Residual FP-SAN Blocks (ternary, per-expert)
    â†’ FP16 PixelShuffle Reconstruction
    â†’ Optional Ternary Refinement
    â†’ Output (HR)

Architecture Decisions Reference:
  - Decision 1: Spike-Routed MoE (architecture_decisions.md)
  - Decision 2: Combinatorial Texture Encoding
  - Decision 3: Layered Spike Precision (ternary + FP16)
  - Decision 4: Dual Homeostatic Control
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

from fpsan_conv2d import FPSANConv2d, FP16Conv2d


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Building Blocks
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

class ChannelAttention(nn.Module):
    """
    Lightweight channel attention via global average pooling.
    
    Computes per-channel importance: GAP â†’ FC(reduce) â†’ ReLU â†’ FC(expand) â†’ Sigmoid
    Cost: O(C) â€” negligible compared to spatial convolutions.
    """

    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        y = self.pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class ResidualFPSANBlock(nn.Module):
    """
    Residual block using FPSANConv2d (ternary quantized).
    
    Structure:
      x â†’ DepthwiseConv(ternary) â†’ ReLU â†’ PointwiseConv(ternary) â†’ ChannelAttn â†’ + x
    
    Depthwise separable design reduces FLOPs by ~8-9Ã— vs standard conv.
    Ternary weights make it multiplication-free at inference.
    """

    def __init__(self, channels: int):
        super().__init__()
        self.depthwise = FPSANConv2d(
            channels, channels, kernel_size=3, padding=1, groups=channels
        )
        self.pointwise = FPSANConv2d(
            channels, channels, kernel_size=1, padding=0
        )
        self.attention = ChannelAttention(channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        out = self.relu(self.depthwise(x))
        out = self.pointwise(out)
        out = self.attention(out)
        return out + residual


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Texture State Encoder (Decision 2: Combinatorial Texture Encoding)
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

class TextureStateEncoder(nn.Module):
    """
    Encodes local texture information into a combinatorial spike chord.
    
    Uses 8 "texture neurons" â€” each fires (>threshold) or doesn't.
    8 neurons = 2^8 - 1 = 255 unique texture states.
    
    Inspired by Broca's vocal chord system (core_dump/FP-SAN-LLM/src/fpsan_v7_broca.cpp)
    but applied to visual texture classification.
    
    Output: soft chord (continuous 0-1 per neuron) for differentiable routing.
    """

    def __init__(self, in_channels: int, num_neurons: int = 8):
        super().__init__()
        self.num_neurons = num_neurons
        # Reduce spatial features to neuron activations
        self.encoder = nn.Sequential(
            FPSANConv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels),
            nn.ReLU(inplace=True),
            FPSANConv2d(in_channels, num_neurons, kernel_size=1, padding=0),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Returns: Texture state activations [B, num_neurons, H, W] in range (0, 1).
        Each spatial position gets its own texture chord.
        """
        return torch.sigmoid(self.encoder(x))


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Spiking Router + Expert System (Decision 1: SR-MoE)
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

class ExpertBlock(nn.Module):
    """Single expert: a small stack of Residual FP-SAN blocks."""

    def __init__(self, channels: int, num_blocks: int = 2):
        super().__init__()
        self.blocks = nn.Sequential(
            *[ResidualFPSANBlock(channels) for _ in range(num_blocks)]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.blocks(x)


class SpikeRoutedMoE(nn.Module):
    """
    Spike-Routed Mixture of Experts.
    
    Instead of softmax routing (dense computation), texture state chords
    are used to compute soft expert weights via learned projections.
    
    During training: soft routing (all experts contribute, weighted).
    During inference: hard routing (top-1 expert only, for speed).
    
    Starting with 4 experts (Decision 1):
      Expert 0: Edge/boundary specialist
      Expert 1: Texture/pattern specialist  
      Expert 2: Flat/gradient specialist
      Expert 3: Fine detail specialist
    
    Experts specialize naturally through gradient flow â€” no manual assignment.
    """

    def __init__(self, channels: int, num_experts: int = 4, blocks_per_expert: int = 2):
        super().__init__()
        self.num_experts = num_experts

        # Expert networks
        self.experts = nn.ModuleList([
            ExpertBlock(channels, blocks_per_expert) for _ in range(num_experts)
        ])

        # Router: texture state neurons â†’ expert weights
        # Input: 8 texture neurons, output: num_experts weights
        self.router = nn.Sequential(
            nn.Conv2d(8, num_experts, kernel_size=1, bias=True),
        )

    def forward(
        self, x: torch.Tensor, texture_state: torch.Tensor
    ) -> torch.Tensor:
        """
        Args:
            x: Feature maps [B, C, H, W]
            texture_state: Texture chord [B, 8, H, W] from TextureStateEncoder
        Returns:
            Routed expert output [B, C, H, W]
        """
        # Compute routing weights from texture state
        # [B, num_experts, H, W]
        route_logits = self.router(texture_state)
        route_weights = torch.softmax(route_logits, dim=1)

        # Soft routing during training: weighted sum of all experts
        if self.training:
            output = torch.zeros_like(x)
            for i, expert in enumerate(self.experts):
                expert_out = expert(x)
                weight = route_weights[:, i:i+1, :, :]  # [B, 1, H, W]
                output = output + expert_out * weight
            return output
        else:
            # Hard routing during inference: top-1 expert only
            top_expert = route_weights.argmax(dim=1)  # [B, H, W]

            output = torch.zeros_like(x)
            for i, expert in enumerate(self.experts):
                mask = (top_expert == i).unsqueeze(1).float()  # [B, 1, H, W]
                if mask.any():
                    expert_out = expert(x)
                    output = output + expert_out * mask
            return output


# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Main Architecture: MYELIN-SRNet
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

class MyelinSRNet(nn.Module):
    """
    MYELIN-SR: Next-Gen Super Resolution powered by FP-SAN.
    
    Architecture:
      1. Shallow Feature Extraction (ternary FPSANConv2d)
      2. Texture State Encoding (combinatorial chord, 255 states)
      3. Spike-Routed MoE (4 specialized experts)
      4. FP16 PixelShuffle Reconstruction (color-precision critical)
      5. Optional Ternary Residual Refinement
    
    Unique capabilities vs DLSS:
      - Continual learning via Astrocytic Stiffness
      - Multiplication-free ternary inference
      - Per-game auto-adaptation
      - 0.1-0.5 MB deploy size
    """

    def __init__(
        self,
        in_channels: int = 3,
        base_channels: int = 48,
        upscale_factor: int = 2,
        num_experts: int = 4,
        blocks_per_expert: int = 2,
        num_texture_neurons: int = 8,
        enable_refinement: bool = True,
    ):
        super().__init__()
        self.upscale_factor = upscale_factor
        self.enable_refinement = enable_refinement

        # â”€â”€ Stage 1: Shallow Feature Extraction (Ternary) â”€â”€
        self.shallow = FPSANConv2d(in_channels, base_channels, kernel_size=3, padding=1)
        self.shallow_relu = nn.ReLU(inplace=True)

        # â”€â”€ Stage 2: Texture State Encoder â”€â”€
        self.texture_encoder = TextureStateEncoder(base_channels, num_texture_neurons)

        # â”€â”€ Stage 3: Spike-Routed MoE â”€â”€
        self.moe = SpikeRoutedMoE(base_channels, num_experts, blocks_per_expert)

        # â”€â”€ Stage 4: FP16 PixelShuffle Reconstruction (Decision 3) â”€â”€
        # This is the color-precision-critical layer â€” uses FP16, not ternary
        upscale_channels = in_channels * (upscale_factor ** 2)
        self.reconstruction = nn.Sequential(
            FP16Conv2d(base_channels, upscale_channels, kernel_size=3, padding=1),
            nn.PixelShuffle(upscale_factor),
        )

        # â”€â”€ Stage 5: Optional Ternary Refinement (Quality Mode) â”€â”€
        if enable_refinement:
            self.refinement = nn.Sequential(
                FPSANConv2d(in_channels, in_channels, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                FPSANConv2d(in_channels, in_channels, kernel_size=3, padding=1),
            )
        else:
            self.refinement = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Bilinear upscale for global residual learning
        x_up = F.interpolate(
            x, scale_factor=self.upscale_factor, mode="bilinear", align_corners=False
        )

        # Stage 1: Extract shallow features (ternary)
        feat = self.shallow_relu(self.shallow(x))

        # Stage 2: Encode texture state (combinatorial chord)
        texture_state = self.texture_encoder(feat)

        # Stage 3: Route through specialized experts
        feat = self.moe(feat, texture_state)

        # Stage 4: FP16 PixelShuffle reconstruction
        sr = self.reconstruction(feat)

        # Global residual: SR = upscaled_input + learned_residual
        sr = sr + x_up

        # Stage 5: Optional refinement pass
        if self.refinement is not None:
            sr = sr + self.refinement(sr)

        return sr

    def consolidate_all(self, rate: float = 0.08) -> None:
        """
        Sleep Consolidation across all FPSANConv2d layers.
        Call after a training phase to protect learned features.
        """
        for module in self.modules():
            if isinstance(module, FPSANConv2d):
                module.consolidate(rate)

    def get_total_sparsity(self) -> float:
        """Average sparsity across all ternary layers."""
        sparsities = []
        for module in self.modules():
            if isinstance(module, FPSANConv2d):
                sparsities.append(module.get_sparsity())
        return sum(sparsities) / max(len(sparsities), 1)

    def get_deploy_size_bytes(self) -> int:
        """Estimated deployment size with ternary packing."""
        ternary_bytes = 0
        fp_bytes = 0
        for name, module in self.named_modules():
            if isinstance(module, FPSANConv2d):
                # 2 bits per ternary weight, packed 16 per uint32
                n_weights = module.weight.numel()
                ternary_bytes += (n_weights * 2 + 31) // 32 * 4  # Pack into uint32s
                if module.bias is not None:
                    fp_bytes += module.bias.numel() * 4  # FP32 biases
                fp_bytes += 4  # Per-layer scale factor
            elif isinstance(module, nn.Conv2d):
                fp_bytes += module.weight.numel() * 2  # FP16 for reconstruction
                if module.bias is not None:
                    fp_bytes += module.bias.numel() * 4
            elif isinstance(module, nn.Linear):
                fp_bytes += module.weight.numel() * 2 + (module.bias.numel() * 4 if module.bias is not None else 0)

        return ternary_bytes + fp_bytes

    def print_architecture_summary(self) -> None:
        """Print a detailed summary of the architecture."""
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        deploy_size = self.get_deploy_size_bytes()
        sparsity = self.get_total_sparsity()

        print("=" * 60)
        print("  MYELIN-SRNet Architecture Summary")
        print("=" * 60)
        print(f"  Upscale factor:    {self.upscale_factor}Ã—")
        print(f"  Total parameters:  {total_params:,}")
        print(f"  Trainable params:  {trainable_params:,}")
        print(f"  Deploy size:       {deploy_size / 1024:.1f} KB ({deploy_size / (1024**2):.3f} MB)")
        print(f"  Ternary sparsity:  {sparsity * 100:.1f}%")
        print(f"  Refinement:        {'ON' if self.enable_refinement else 'OFF'}")
        print(f"  Experts:           {self.moe.num_experts}")
        print("=" * 60)


def build_myelin_sr(
    upscale: int = 2,
    quality_mode: str = "quality",
) -> MyelinSRNet:
    """
    Factory function with DLSS-style quality presets.
    
    Presets:
      - "performance": No refinement, 2 experts, minimal blocks. Fastest.
      - "balanced":    No refinement, 4 experts. Good trade-off.
      - "quality":     Full refinement, 4 experts. Best quality.
      - "ultra":       Full refinement, 4 experts, wider channels. Maximum quality.
    """
    configs = {
        "performance": dict(base_channels=32, num_experts=2, blocks_per_expert=1, enable_refinement=False),
        "balanced":    dict(base_channels=48, num_experts=4, blocks_per_expert=2, enable_refinement=False),
        "quality":     dict(base_channels=48, num_experts=4, blocks_per_expert=2, enable_refinement=True),
        "ultra":       dict(base_channels=64, num_experts=4, blocks_per_expert=3, enable_refinement=True),
    }

    if quality_mode not in configs:
        raise ValueError(f"Unknown quality mode: {quality_mode}. Choose from {list(configs.keys())}")

    return MyelinSRNet(upscale_factor=upscale, **configs[quality_mode])


if __name__ == "__main__":
    # Quick sanity check
    model = build_myelin_sr(upscale=2, quality_mode="quality")
    model.print_architecture_summary()

    # Test forward pass
    dummy_lr = torch.randn(1, 3, 64, 64)
    dummy_hr = model(dummy_lr)
    print(f"\n  Input:  {dummy_lr.shape}")
    print(f"  Output: {dummy_hr.shape}")
    assert dummy_hr.shape == (1, 3, 128, 128), f"Expected (1,3,128,128), got {dummy_hr.shape}"
    print("  âœ… Forward pass OK")


## Phase 2 Architecture: MYELIN-Flow & TemporalFusion

In [ ]:
"""
MYELIN-Flow: Lightweight Ternary Optical Flow Estimator

Traditional flow models (RAFT, FlowNet) use millions of FP32 parameters and
dense O(N^2) correlation volumes. This is too heavy for a DLSS replacement.

MYELIN-Flow uses our FP-SAN architecture to build a fast, ternary motion 
estimator specifically designed to provide flow vectors for TemporalFusion.

Key Innovations:
1. TERNARY FEATURE PYRAMID: Extracts 3 scales of features using {-1, 0, 1} math.
2. SPIKE-ROUTED CORRELATION: Since features are ternary, they form highly sparse
   spike maps. Instead of computing dense correlation across all pixels, we mask
   the correlation volume using the spikes. Zero * anything = 0, so we skip it.
   This provides ~70% math sparsity during inference.
3. GRU-FREE DECODER: Employs a lightweight feed-forward decoder instead of expensive
   iterative RNN refinement.
"""


class SpikeRoutedCorrelation(nn.Module):
    """
    Novel ternary correlation volume.
    
    Standard optical flow computes a dot product between every pixel feature
    in Frame T and its neighborhood in Frame T-1.
    
    Since our features are ternary and highly sparse, we compute a "Spike Mask".
    We only correlate where there is active signal.
    """
    def __init__(self, search_radius: int = 4):
        super().__init__()
        self.radius = search_radius
        self.out_channels = (2 * search_radius + 1) ** 2
        
    def forward(self, f1: torch.Tensor, f2: torch.Tensor) -> torch.Tensor:
        """
        f1: features from Frame T
        f2: features from Frame T-1
        Returns: correlation volume [B, (2r+1)^2, H, W]
        """
        B, C, H, W = f1.shape
        
        # Determine sparse active regions (magnitude > 0.1 to ignore noise)
        # In ternary logic, valid features are 1.0 or -1.0
        mask_f1 = (f1.abs().sum(dim=1, keepdim=True) > 0.1).float()
        
        corr_volume = []
        for dy in range(-self.radius, self.radius + 1):
            for dx in range(-self.radius, self.radius + 1):
                # Shift f2 by (dx, dy)
                # Pad to keep output same size
                pad_left = max(dx, 0)
                pad_right = max(-dx, 0)
                pad_top = max(dy, 0)
                pad_bottom = max(-dy, 0)
                
                f2_shifted = F.pad(f2, (pad_left, pad_right, pad_top, pad_bottom))
                # Crop back to H, W
                start_y = pad_top
                start_x = pad_left
                f2_cropped = f2_shifted[:, :, start_y:start_y+H, start_x:start_x+W]
                
                # Dot product along channel dimension
                corr = (f1 * f2_cropped).sum(dim=1, keepdim=True)
                
                # Spike-routing mask: eliminate calculations in dead zones
                corr = corr * mask_f1
                # Normalize by channel count
                corr = corr / float(C)
                
                corr_volume.append(corr)
                
        return torch.cat(corr_volume, dim=1)


class FlowEncoder(nn.Module):
    """
    Ternary Feature Pyramid Extractor.
    Extracts features at 1/2, 1/4, 1/8 resolution.
    """
    def __init__(self):
        super().__init__()
        # Input 3ch RGB -> 16ch features at 1/2 resolution
        self.conv1 = FPSANConv2d(3, 16, kernel_size=7, stride=2, padding=3)
        self.conv2 = FPSANConv2d(16, 16, kernel_size=3, stride=1, padding=1)
        
        # 16ch -> 24ch at 1/4 resolution
        self.conv3 = FPSANConv2d(16, 24, kernel_size=3, stride=2, padding=1)
        self.conv4 = FPSANConv2d(24, 24, kernel_size=3, stride=1, padding=1)
        
        # 24ch -> 32ch at 1/8 resolution
        self.conv5 = FPSANConv2d(24, 32, kernel_size=3, stride=2, padding=1)
        self.conv6 = FPSANConv2d(32, 32, kernel_size=3, stride=1, padding=1)
        
        self.act = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        f2 = self.act(self.conv2(self.act(self.conv1(x))))
        f4 = self.act(self.conv4(self.act(self.conv3(f2))))
        f8 = self.act(self.conv6(self.act(self.conv5(f4))))
        return f2, f4, f8


class FlowDecoderLevel(nn.Module):
    """
    Decodes flow at a specific pyramid level.
    Inputs:
    - corr: Correlation volume [B, 81, H, W]
    - f1: Features of Frame 1 at this level [B, C, H, W]
    - up_flow: Upsampled flow from previous level [B, 2, H, W] (if any)
    """
    def __init__(self, in_channels: int):
        super().__init__()
        # In = corr_channels (81) + feat_channels + up_flow (2)
        # Out = 2 (flow x,y)
        self.conv1 = FPSANConv2d(in_channels, 64, kernel_size=3, padding=1)
        self.conv2 = FPSANConv2d(64, 32, kernel_size=3, padding=1)
        self.pred  = FPSANConv2d(32, 2, kernel_size=3, padding=1)
        self.act   = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        return self.pred(x)


class MyelinFlow(nn.Module):
    """
    Complete MYELIN-Flow temporal backbone.
    Parameters targeting < 50K. Pure ternary forward pass natively compatible
    with DX12 multiplier-free compute.
    """
    def __init__(self, search_radius: int = 4):
        super().__init__()
        self.encoder = FlowEncoder()
        
        self.corr_8 = SpikeRoutedCorrelation(search_radius)
        self.corr_4 = SpikeRoutedCorrelation(search_radius)
        
        corr_ch = self.corr_8.out_channels # (2r+1)^2 = 81
        
        # Decoder levels
        # Level 8: decodes 1/8 scale flow
        self.dec_8 = FlowDecoderLevel(corr_ch + 32)
        
        # Level 4: decodes 1/4 scale flow
        self.dec_4 = FlowDecoderLevel(corr_ch + 24 + 2)
        
        # Level 2 upsampler
        self.upflow_2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            FPSANConv2d(2, 2, kernel_size=3, padding=1) # Smooth upsampled flow
        )
        
    def _warp(self, x: torch.Tensor, flow: torch.Tensor) -> torch.Tensor:
        """
        Warps tensor x using optical flow.
        Flow is in pixel coordinates.
        """
        B, C, H, W = x.shape
        # Create meshgrid
        mesh_x, mesh_y = torch.meshgrid(torch.arange(W), torch.arange(H), indexing='xy')
        mesh_x = mesh_x.to(x.device).float().expand(B, -1, -1)
        mesh_y = mesh_y.to(x.device).float().expand(B, -1, -1)
        
        # Apply flow
        u = flow[:, 0, :, :]
        v = flow[:, 1, :, :]
        new_x = mesh_x + u
        new_y = mesh_y + v
        
        # Normalize to [-1, 1] for grid_sample
        new_x = 2.0 * new_x / max(W - 1, 1) - 1.0
        new_y = 2.0 * new_y / max(H - 1, 1) - 1.0
        grid = torch.stack((new_x, new_y), dim=-1)
        
        return F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=True)

    def forward(self, img1: torch.Tensor, img2: torch.Tensor):
        """
        img1: Frame T (Current)
        img2: Frame T-1 (Previous)
        Returns: Flow field from T -> T-1 [B, 2, H, W]
        """
        # 1. Feature Extraction
        f2_1, f4_1, f8_1 = self.encoder(img1)
        f2_2, f4_2, f8_2 = self.encoder(img2)
        
        # 2. Level 8 (1/8 resolution)
        corr8 = self.corr_8(f8_1, f8_2)
        x8 = torch.cat([corr8, f8_1], dim=1)
        flow8 = self.dec_8(x8)
        
        # 3. Level 4 (1/4 resolution)
        # Upsample flow from level 8 to warp level 4 features
        up_flow8 = F.interpolate(flow8, scale_factor=2, mode='bilinear', align_corners=False) * 2.0
        warp_f4_2 = self._warp(f4_2, up_flow8)
        
        # Compute correlation with warped previous frame (residual flow)
        corr4 = self.corr_4(f4_1, warp_f4_2)
        x4 = torch.cat([corr4, f4_1, up_flow8], dim=1)
        res_flow4 = self.dec_4(x4)
        
        flow4 = up_flow8 + res_flow4
        
        # 4. Final Upsample to Full Resolution
        # Because we do super-resolution, we upsample flow4 x4 to reach original RGB input size
        flow_full = F.interpolate(flow4, scale_factor=4, mode='bilinear', align_corners=False) * 4.0
        
        return flow_full

    def print_architecture_summary(self):
        total_params = sum(p.numel() for p in self.parameters())
        ternary_params = sum(p.numel() for m in self.modules() if isinstance(m, FPSANConv2d) for p in m.parameters())
        print("=" * 60)
        print("  MYELIN-Flow Architecture Summary")
        print("=" * 60)
        print(f"  Total parameters:  {total_params:,}")
        print(f"  Ternary backbone:  {ternary_params/total_params*100:.1f}% ternary")
        print(f"  Deploy size:       {(ternary_params/4 + (total_params-ternary_params)*4) / 1024:.1f} KB")
        print("=" * 60)


In [ ]:
"""
MYELIN-Temporal: Deep Temporal Accumulation for MYELIN-SR

This module fuses the current SR output with the accumulated history from
previous frames. It uses MYELIN-Flow to align the history to the current frame.

Key Innovations:
1. ASTROCYTIC TEMPORAL GATING: Instead of a hard-coded alpha blend factor
   like TAA, we use a single FPSANConv2d layer to learn the blend weight.
   This leverages Astrocytic Stiffness â€” static regions freeze the weights
   to output high history dependence, while dynamic regions stay plastic
   and favor the current frame.
2. DISOCCLUSION DETECTION: History is discarded when the flow warp is invalid
   (e.g., when a foreground object moves, revealing a previously hidden background).
"""


class TemporalFusion(nn.Module):
    """
    Fuses the high-resolution output of the current frame with the history.
    """
    def __init__(self, in_channels: int = 3, flow_search_radius: int = 4):
        super().__init__()
        
        # The lightweight optical flow backbone
        self.flow_estimator = MyelinFlow(search_radius=flow_search_radius)
        
        # Disocclusion Head: Detects when to completely discard history
        self.disocclusion_conv = FPSANConv2d(in_channels * 2 + 2, 16, kernel_size=3, padding=1)
        self.disocclusion_out  = FPSANConv2d(16, 1, kernel_size=3, padding=1)
        
        # Astrocytic Temporal Gating: Learns the blending alpha
        self.gating = FPSANConv2d(in_channels * 2 + 2, 1, kernel_size=3, padding=1)
        
    def _warp(self, x: torch.Tensor, flow: torch.Tensor) -> torch.Tensor:
        """
        Warps tensor x using optical flow.
        Flow is in pixel coordinates.
        """
        B, C, H, W = x.shape
        mesh_x, mesh_y = torch.meshgrid(torch.arange(W), torch.arange(H), indexing='xy')
        mesh_x = mesh_x.to(x.device).float().expand(B, -1, -1)
        mesh_y = mesh_y.to(x.device).float().expand(B, -1, -1)
        
        u = flow[:, 0, :, :]
        v = flow[:, 1, :, :]
        new_x = mesh_x + u
        new_y = mesh_y + v
        
        new_x = 2.0 * new_x / max(W - 1, 1) - 1.0
        new_y = 2.0 * new_y / max(H - 1, 1) - 1.0
        grid = torch.stack((new_x, new_y), dim=-1)
        
        return F.grid_sample(x, grid, mode='bilinear', padding_mode='zeros', align_corners=True)

    def forward(
        self, 
        sr_current: torch.Tensor, 
        sr_history: torch.Tensor, 
        lr_current: torch.Tensor, 
        lr_history: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        sr_current: SR output for Frame t
        sr_history: Accumulated SR output from Frame t-1
        lr_current: Low-res Frame t (to estimate flow)
        lr_history: Low-res Frame t-1 (to estimate flow)
        
        Returns:
            fused_sr: Chronologically stable SR frame
            flow: Computed flow (for loss supervision)
        """
        B, C, H, W = sr_current.shape
        
        # 1. Estimate Flow (computed on low-res input for speed)
        # Flow vectors match the LR scale. We must upsample and scale them for HR.
        flow_lr = self.flow_estimator(lr_current, lr_history)
        scale_factor = float(sr_current.shape[-1]) / lr_current.shape[-1]
        
        flow_hr = F.interpolate(flow_lr, size=(H, W), mode='bilinear', align_corners=False)
        flow_hr = flow_hr * scale_factor
        
        # 2. Warp History SR to Current Frame Alignment
        warped_history = self._warp(sr_history, flow_hr)
        
        # 3. Disocclusion & Blending Features
        # Concat: Current SR + Warped History SR + HR Flow
        features = torch.cat([sr_current, warped_history, flow_hr], dim=1)
        
        # 4. Predict Disocclusion Mask [0, 1]
        # 1.0 = occluded (discard history), 0.0 = valid history
        d_mask = F.leaky_relu(self.disocclusion_conv(features), 0.1)
        d_mask = torch.sigmoid(self.disocclusion_out(d_mask))
        
        # 5. Astrocytic Temporal Gating (Alpha Blend)
        # 1.0 = use only current, 0.0 = use only history
        alpha = torch.sigmoid(self.gating(features))
        
        # 6. Apply Disocclusion: force alpha=1.0 where history is invalid
        alpha = torch.max(alpha, d_mask)
        
        # 7. Final Temporal Blend
        fused_sr = alpha * sr_current + (1.0 - alpha) * warped_history
        
        return fused_sr, flow_hr

    def print_architecture_summary(self):
        self.flow_estimator.print_architecture_summary()
        
        tp = sum(p.numel() for p in self.parameters())
        fp = sum(p.numel() for p in self.flow_estimator.parameters())
        fusion_params = tp - fp
        
        print(f"  Temporal Fusion Params: {fusion_params:,}")
        print(f"  Total Temporal Subsystem: {tp:,}")
        print("=" * 60)


## Datasets & Losses

In [ ]:
"""
Loss functions for MYELIN-SR training.

Round 2 improvements â€” 3 losses working together:

  1. CHARBONNIER LOSS (replaces L1):
     sqrt((x-y)^2 + epsilon^2) â€” smoothed L1 that handles outlier pixels better.
     Does not saturate at large errors like MSE, not noisy at small errors like L1.

  2. FFT FREQUENCY LOSS (novel addition):
     Takes 2D FFT of SR and HR outputs, penalizes the *frequency spectrum difference*.
     L1-based pixel loss ignores high-frequency texture patterns completely.
     FFT loss directly supervises the network on sharpness and edge information â€”
     this is exactly where SR models fail (blurry output = missing high frequencies).

  3. VGG PERCEPTUAL LOSS (retained from Round 1):
     High-level semantic feature matching. Complements FFT (which is low-level).

  Why ternary networks benefit especially from FFT loss:
     With {-1, 0, +1} weights, each neuron responds to patterns via add/subtract.
     FFT loss creates gradient pressure toward high-frequency feature detectors,
     which naturally maps to ON-OFF center-surround patterns â€” the exact structure
     ternary {-1,+1} weights produce. It's a loss designed for our architecture.
"""



class VGGPerceptualLoss(nn.Module):
    """
    Perceptual loss using VGG19 features.

    Compares high-level features between SR output and HR ground truth.
    Uses relu3_3 features (layer index 16) â€” balances texture and structure.

    Frozen VGG (no gradients) â€” only used as a feature extractor.
    """

    def __init__(self, feature_layer: int = 16):
        super().__init__()
        vgg = models.vgg19(weights=VGG19_Weights.IMAGENET1K_V1).features[:feature_layer + 1]

        # Freeze all VGG parameters
        for param in vgg.parameters():
            param.requires_grad = False

        self.vgg = vgg
        self.register_buffer(
            "mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )
        self.register_buffer(
            "std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

    def _normalize(self, x: torch.Tensor) -> torch.Tensor:
        """Normalize to ImageNet statistics."""
        return (x - self.mean) / self.std

    def forward(self, sr: torch.Tensor, hr: torch.Tensor) -> torch.Tensor:
        sr_features = self.vgg(self._normalize(sr.clamp(0, 1)))
        hr_features = self.vgg(self._normalize(hr.clamp(0, 1)))
        return F.l1_loss(sr_features, hr_features.detach())


class CharbonnierLoss(nn.Module):
    """
    Charbonnier Loss: smoothed L1 that behaves like L2 for small errors,
    L1 for large errors.

    L_charb(x, y) = sqrt((x-y)^2 + epsilon^2)

    Why better than L1 for SR:
    - L1 has a gradient discontinuity at 0 (causes training instability for ternary)
    - Charbonnier is smooth everywhere -> better gradient signal for quantized networks
    - Slightly more robust to outlier pixels than MSE
    """

    def __init__(self, epsilon: float = 1e-3):
        super().__init__()
        self.epsilon_sq = epsilon ** 2

    def forward(self, sr: torch.Tensor, hr: torch.Tensor) -> torch.Tensor:
        diff_sq = (sr - hr) ** 2
        return torch.mean(torch.sqrt(diff_sq + self.epsilon_sq))


class FFTFrequencyLoss(nn.Module):
    """
    Frequency Spectrum Loss via 2D FFT.

    Key insight: Standard pixel losses (L1/L2) are equally weighted across all
    spatial frequencies. But SR is about recovering HIGH frequencies â€” fine
    texture, sharp edges, and detail.

    This loss computes the L1 difference of the 2D FFT magnitude spectrum,
    directly supervising the network to produce correct frequency content.

    Works in YCbCr space (Y channel only) to focus on luminance structure.

    Novel aspect: Exponential high-frequency emphasis â€” weight the spectrum
    inverse to frequency magnitude so rare high-freq components get more gradient.
    """

    def __init__(
        self,
        hf_emphasis: float = 2.0,   # exponent for high-frequency weighting
        use_log: bool = True,        # log-magnitude spectrum is more perceptually uniform
    ):
        super().__init__()
        self.hf_emphasis = hf_emphasis
        self.use_log = use_log

    def _rgb_to_y(self, img: torch.Tensor) -> torch.Tensor:
        """Extract luminance channel (Y) from RGB."""
        r, g, b = img[:, 0:1], img[:, 1:2], img[:, 2:3]
        return 0.2126 * r + 0.7152 * g + 0.0722 * b

    def _make_hf_weight(self, h: int, w: int, device: torch.device) -> torch.Tensor:
        """
        Create 2D spatial frequency weight map.
        Higher frequencies (closer to edges of FFT) get higher weight.
        """
        fy = torch.fft.fftfreq(h, device=device).abs()
        fx = torch.fft.fftfreq(w, device=device).abs()
        # Outer product: each position gets magnitude of its frequency
        freq_map = torch.sqrt(fy.unsqueeze(1) ** 2 + fx.unsqueeze(0) ** 2)
        # Normalize and apply emphasis
        freq_map = freq_map / freq_map.max().clamp_min(1e-8)
        weight = (1.0 + freq_map) ** self.hf_emphasis
        return weight.unsqueeze(0).unsqueeze(0)   # [1, 1, H, W]

    def forward(self, sr: torch.Tensor, hr: torch.Tensor) -> torch.Tensor:
        sr_y = self._rgb_to_y(sr.clamp(0, 1))
        hr_y = self._rgb_to_y(hr.clamp(0, 1))

        H, W = sr_y.shape[-2], sr_y.shape[-1]

        # 2D FFT
        sr_fft = torch.fft.fft2(sr_y, norm="ortho")
        hr_fft = torch.fft.fft2(hr_y, norm="ortho")

        # Magnitude spectrum
        sr_mag = sr_fft.abs()
        hr_mag = hr_fft.abs()

        if self.use_log:
            sr_mag = torch.log1p(sr_mag)
            hr_mag = torch.log1p(hr_mag)

        # Weighted L1 on frequency spectrum
        weight = self._make_hf_weight(H, W, sr.device)
        loss = (weight * (sr_mag - hr_mag).abs()).mean()
        return loss


class MyelinSRLoss(nn.Module):
    """
    Combined training loss for MYELIN-SR Round 2.

    Total = Charbonnier + Î»_freq Ã— FFTFrequency + Î»_perceptual Ã— Perceptual

    Round 1 formula:  L1 + Î»_perceptual Ã— Perceptual
    Round 2 formula:  Charbonnier + Î»_freq Ã— FFT + Î»_perceptual Ã— Perceptual

    The FFT loss is the key new addition â€” it directly teaches the network to
    produce correct high-frequency content, which is exactly what SR needs.
    """

    def __init__(
        self,
        perceptual_weight: float = 0.1,
        freq_weight: float = 0.05,
        charbonnier_epsilon: float = 1e-3,
        use_perceptual: bool = True,
        use_freq: bool = True,
    ):
        super().__init__()
        self.pixel_loss      = CharbonnierLoss(epsilon=charbonnier_epsilon)
        self.perceptual_weight = perceptual_weight
        self.freq_weight     = freq_weight
        self.use_perceptual  = use_perceptual
        self.use_freq        = use_freq

        if use_perceptual:
            self.perceptual_loss = VGGPerceptualLoss()
        else:
            self.perceptual_loss = None

        if use_freq:
            self.freq_loss = FFTFrequencyLoss(hf_emphasis=2.0, use_log=True)
        else:
            self.freq_loss = None

    def forward(
        self, sr: torch.Tensor, hr: torch.Tensor
    ) -> tuple[torch.Tensor, dict]:
        """
        Returns:
            total_loss: Combined scalar loss for backward pass
            loss_dict:  Individual loss components for logging
        """
        pixel = self.pixel_loss(sr, hr)
        loss_dict = {"charbonnier": pixel.item()}

        total = pixel

        if self.use_freq and self.freq_loss is not None:
            freq = self.freq_loss(sr, hr)
            total = total + self.freq_weight * freq
            loss_dict["fft_freq"] = freq.item()

        if self.use_perceptual and self.perceptual_loss is not None:
            perceptual = self.perceptual_loss(sr, hr)
            total = total + self.perceptual_weight * perceptual
            loss_dict["perceptual"] = perceptual.item()

        loss_dict["total"] = total.item()
        return total, loss_dict


"""
Temporal Loss Functions for MYELIN-SR Phase 2

These losses supervise the MYELIN-Flow network and enforce temporal consistency
between consecutive super-resolved frames to prevent shimmering/flickering.
"""


def warp_image(x: torch.Tensor, flow: torch.Tensor) -> torch.Tensor:
    """
    Warp an image using dense optical flow vectors.
    flow: [B, 2, H, W] in pixel displacement
    """
    B, C, H, W = x.shape
    mesh_x, mesh_y = torch.meshgrid(torch.arange(W), torch.arange(H), indexing='xy')
    mesh_x = mesh_x.to(x.device).float().expand(B, -1, -1)
    mesh_y = mesh_y.to(x.device).float().expand(B, -1, -1)
    
    u = flow[:, 0, :, :]
    v = flow[:, 1, :, :]
    new_x = mesh_x + u
    new_y = mesh_y + v
    
    # Normalize to [-1, 1] for grid_sample
    new_x = 2.0 * new_x / max(W - 1, 1) - 1.0
    new_y = 2.0 * new_y / max(H - 1, 1) - 1.0
    grid = torch.stack((new_x, new_y), dim=-1)
    
    return F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=True)

class MyelinTemporalLoss(nn.Module):
    """
    Joint loss for Spatial SR + Temporal Consistency + Optical Flow.
    """
    def __init__(
        self, 
        spatial_weight: float = 1.0,
        temp_consistency_weight: float = 0.5,
        flow_warp_weight: float = 1.0,
    ):
        super().__init__()
        self.w_spatial = spatial_weight
        self.w_temp = temp_consistency_weight
        self.w_flow = flow_warp_weight
        
        self.charbonnier = CharbonnierLoss(epsilon=1e-3)
        self.fft = FFTFrequencyLoss()
        
    def forward(
        self, 
        sr_t: torch.Tensor, 
        sr_t_minus_1: torch.Tensor, 
        hr_t: torch.Tensor, 
        hr_t_minus_1: torch.Tensor,
        flow_hr: torch.Tensor
    ):
        """
        sr_t: Current frame SR output
        sr_t_minus_1: Previous frame SR output
        hr_t: Current ground truth HR
        hr_t_minus_1: Previous ground truth HR
        flow_hr: Estimated flow (upscaled to HR resolution)
        """
        # 1. Spatial SR Loss (Phase 1 standard)
        l_spatial_pixel = self.charbonnier(sr_t, hr_t)
        l_spatial_freq = self.fft(sr_t, hr_t)
        l_spatial = l_spatial_pixel + 0.05 * l_spatial_freq
        
        # 2. Flow Warp Loss (Supervises MYELIN-Flow unsupervisedly)
        # If flow is correct, warping HR_t-1 should look exactly like HR_t
        warped_hr = warp_image(hr_t_minus_1, flow_hr)
        l_flow = self.charbonnier(warped_hr, hr_t)
        
        # 3. Temporal Consistency Loss (Supervises the TemporalFusion blend)
        # Prevents flickering: the fused SR_t should naturally align with warped SR_t-1
        warped_sr = warp_image(sr_t_minus_1, flow_hr)
        l_temp = self.charbonnier(sr_t, warped_sr)
        
        # Total Joint Loss
        total_loss = (self.w_spatial * l_spatial) + \
                     (self.w_flow * l_flow) + \
                     (self.w_temp * l_temp)
                     
        loss_dict = {
            "spatial": l_spatial.item(),
            "flow_warp": l_flow.item(),
            "temporal_consistency": l_temp.item(),
            "total": total_loss.item()
        }
        
        return total_loss, loss_dict


In [ ]:
"""
REDS (REalistic and Diverse Scenes) Dataset Loader for Phase 2 Temporal Training.

Unlike DIV2K which is static images, REDS is video sequences.
We must load pairs of consecutive frames: (Frame T-1, Frame T).
"""


class REDSSequenceDataset(Dataset):
    """
    Loads consecutive frame pairs (T-1, T) from the REDS dataset.
    Provides data augmentation (flips, rotations, crops) consistently across pairs
    to ensure optical flow vectors remain valid during training.
    """
    def __init__(
        self, 
        root_dir: str, 
        split: str = 'train', 
        scale: int = 2, 
        patch_size: int = 128
    ):
        self.root = Path(root_dir) / split
        self.hr_dir = self.root / 'train_sharp'
        self.scale = scale
        self.patch_size = patch_size
        
        # REDS typically provides X4 bicubic. We will generate our own X2
        # dynamically or use the provided X4 if scale == 4.
        
        self.sequences = []
        
        # Each folder under train_sharp is a sequence (000 to 239 for train)
        if self.hr_dir.exists():
            for seq_folder in sorted(os.listdir(self.hr_dir)):
                seq_path = self.hr_dir / seq_folder
                if not seq_path.is_dir(): continue
                
                # Frames are numbered 00000000.png to 00000099.png
                frames = sorted([f for f in os.listdir(seq_path) if f.endswith('.png')])
                
                # Create overlapping pairs: (0,1), (1,2), (2,3)...
                for i in range(len(frames) - 1):
                    self.sequences.append({
                        'seq': seq_folder,
                        't_minus_1': frames[i],
                        't': frames[i+1]
                    })

    def __len__(self):
        return len(self.sequences)

    def _load_img(self, path) -> torch.Tensor:
        img = Image.open(path).convert('RGB')
        return TF.to_tensor(img)

    def __getitem__(self, idx):
        item = self.sequences[idx]
        seq_folder = item['seq']
        
        hr_path_prev = self.hr_dir / seq_folder / item['t_minus_1']
        hr_path_cur  = self.hr_dir / seq_folder / item['t']
        
        hr_prev = self._load_img(hr_path_prev)
        hr_cur  = self._load_img(hr_path_cur)
        
        # Consistent Random Cropping across the sequence
        _, H, W = hr_prev.shape
        # Ensure patch is an even multiple of scale
        hr_patch = self.patch_size * self.scale
        
        if H > hr_patch and W > hr_patch:
            top  = random.randint(0, H - hr_patch)
            left = random.randint(0, W - hr_patch)
            
            hr_prev = TF.crop(hr_prev, top, left, hr_patch, hr_patch)
            hr_cur  = TF.crop(hr_cur,  top, left, hr_patch, hr_patch)
        else:
            # Center crop if too small
            hr_prev = TF.center_crop(hr_prev, [hr_patch, hr_patch])
            hr_cur  = TF.center_crop(hr_cur,  [hr_patch, hr_patch])
            
        # Create LR images via bicubic downsampling
        # This guarantees perfect alignment for flow training
        lr_size = [self.patch_size, self.patch_size]
        lr_prev = F.interpolate(hr_prev.unsqueeze(0), size=lr_size, mode='bicubic', align_corners=False).squeeze(0)
        lr_cur  = F.interpolate(hr_cur.unsqueeze(0),  size=lr_size, mode='bicubic', align_corners=False).squeeze(0)
        
        # Consistent Data Augmentation (H-flip, V-flip)
        if random.random() < 0.5:
            hr_prev, hr_cur = TF.hflip(hr_prev), TF.hflip(hr_cur)
            lr_prev, lr_cur = TF.hflip(lr_prev), TF.hflip(lr_cur)
        
        if random.random() < 0.5:
            hr_prev, hr_cur = TF.vflip(hr_prev), TF.vflip(hr_cur)
            lr_prev, lr_cur = TF.vflip(lr_prev), TF.vflip(lr_cur)
            
        # Return as two tuples: (LR_prev, LR_cur) and (HR_prev, HR_cur)
        return torch.stack([lr_prev, lr_cur]), torch.stack([hr_prev, hr_cur])

def build_temporal_loader(root_dir: str, scale: int, patch_size: int, batch_size: int, num_workers: int = 4):
    dataset = REDSSequenceDataset(root_dir, scale=scale, patch_size=patch_size)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)


## Training Loop

In [ ]:
def train_temporal(config):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Starting Temporal Training on: {device}")
    
    # 1. Load Pretrained Spatial Engine (Phase 1)
    print(f"Loading Phase 1 Spatial Backbone from {config['spatial_ckpt']}...")
    spatial_model_raw = build_myelin_sr(config['scale'], 'quality')
    
    ckpt = torch.load(config['spatial_ckpt'], map_location='cpu', weights_only=False)
    if 'model_state' in ckpt:
        # Load state dictated by the checkpoint format used in Kaggle train
        # Strip 'module.' prefix if it was saved via DataParallel
        state = {k.replace('module.', ''): v for k,v in ckpt['model_state'].items()}
        spatial_model_raw.load_state_dict(state)
    
    spatial_model_raw.eval()
    for param in spatial_model_raw.parameters():
        param.requires_grad = False  # Freeze spatial engine
        
    spatial_model = nn.DataParallel(spatial_model_raw).to(device)
    print("Spatial backbone loaded and frozen.")
    
    # 2. Initialize Temporal Fusion Engine (Phase 2)
    fusion_model = TemporalFusion(flow_search_radius=config['flow_radius']).to(device)
    fusion_model = nn.DataParallel(fusion_model) if torch.cuda.device_count() > 1 else fusion_model
    fusion_model.train()
    
    # 3. Optimizers & Losses
    optimizer = torch.optim.AdamW(fusion_model.parameters(), lr=config['base_lr'], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, verbose=True
    )
    
    criterion = MyelinTemporalLoss(
        spatial_weight=config['spatial_weight'],
        temp_consistency_weight=config['temporal_weight'],
        flow_warp_weight=config['flow_weight']
    ).to(device)
    
    # 4. Dataset
    print(f"Loading REDS Sequence Dataset from {config['data_dir']}...")
    train_loader = build_temporal_loader(
        root_dir=config['data_dir'],
        scale=config['scale'],
        patch_size=config['patch_size'],
        batch_size=config['batch_size'],
        num_workers=config['num_workers']
    )
    
    # 5. Training Loop
    best_loss = float('inf')
    ckpt_dir = Path("/kaggle/working/temporal_checkpoints")
    ckpt_dir.mkdir(exist_ok=True, parents=True)
    
    for epoch in range(1, config['epochs'] + 1):
        ep_loss = 0.0
        ep_temp_loss = 0.0
        n_batches = 0
        
        for lr_seq, hr_seq in tqdm(train_loader, desc=f"Epoch {epoch:03d}"):
            # Sequences are stacked: [LR_prev, LR_cur]
            lr_prev = lr_seq[:, 0].to(device)
            lr_cur  = lr_seq[:, 1].to(device)
            
            hr_prev = hr_seq[:, 0].to(device)
            hr_cur  = hr_seq[:, 1].to(device)
            
            # Step A: Compute Spatial Output (Frozen)
            with torch.no_grad():
                sr_prev_base = spatial_model(lr_prev)
                sr_cur_base  = spatial_model(lr_cur)
                
            # Step B: Temporal Fusion & Flow Warp
            optimizer.zero_grad()
            fused_sr, flow_hr = fusion_model(sr_cur_base, sr_prev_base, lr_cur, lr_prev)
            
            # Step C: Compute Joint Loss
            loss, l_dict = criterion(fused_sr, sr_prev_base, hr_cur, hr_prev, flow_hr)
            
            loss.backward()
            nn.utils.clip_grad_norm_(fusion_model.parameters(), 1.0)
            optimizer.step()
            
            ep_loss += loss.item()
            ep_temp_loss += l_dict['temporal_consistency']
            n_batches += 1
            
        avg_loss = ep_loss / max(n_batches, 1)
        avg_temp = ep_temp_loss / max(n_batches, 1)
        
        print(f"Epoch {epoch:03d} | Total Loss: {avg_loss:.4f} | Temp Loss: {avg_temp:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        scheduler.step(avg_loss)
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            f_obj = fusion_model.module if isinstance(fusion_model, nn.DataParallel) else fusion_model
            torch.save({
                'epoch': epoch,
                'fusion_state': f_obj.state_dict(),
                'best_loss': best_loss,
            }, ckpt_dir / 'temporal_fusion_best.pt')
            print(f"  >> New Best Temporal Loss: {best_loss:.4f}")

if __name__ == "__main__":
    CONFIG = {
        'scale': 2,
        'epochs': 150,
        'batch_size': 8,
        'patch_size': 128,  # HR patch will be 256
        'num_workers': 4,
        'base_lr': 2e-4,
        'flow_radius': 4,
        'spatial_weight': 1.0,
        'temporal_weight': 0.5,
        'flow_weight': 1.0,
        'spatial_ckpt': 'outputs/myelin_sr_best.pt', # Generated from Phase 1
        'data_dir': '/kaggle/input/reds-dataset',
    }
    
    # Normally started from Kaggle Notebook, this allows direct debug
    # train_temporal(CONFIG)


## Execution

In [ ]:
# PATH CONFIGURATION
# 1. You must add the downloaded Phase 1 output (myelin_sr_deploy.bin or myelin_sr_best.pt) to Kaggle
# 2. You must add the REDS dataset to Kaggle

# Example paths (adjust based on your Kaggle dataset names):
SPATIAL_CKPT = "/kaggle/input/myelin-sr-phase1/myelin_sr_best.pt"
REDS_DIR = "/kaggle/input/reds-dataset"

# Wait for Kaggle to connect to the dataset
import time
while not Path(SPATIAL_CKPT).exists() or not Path(REDS_DIR).exists():
    print("Waiting for paths to become available...")
    print(f"Checking {SPATIAL_CKPT}")
    print(f"Checking {REDS_DIR}")
    time.sleep(5)

CONFIG = {
    'scale': 2,
    'epochs': 150,
    'batch_size': 16, # Adjust based on 16GB T4 VRAM
    'patch_size': 128, 
    'num_workers': 4,
    'base_lr': 2e-4,
    'flow_radius': 4,
    'spatial_weight': 1.0,
    'temporal_weight': 0.5,
    'flow_weight': 1.0,
    'spatial_ckpt': SPATIAL_CKPT,
    'data_dir': REDS_DIR,
}

print("Starting Temporal Pipeline...")
train_temporal(CONFIG)
